# Weekly Study File Archive

*Combine directory setup and file I/O by saving and reopening one small study archive in text, CSV, JSON, pickle, and image formats.*

This integrated exercise emphasizes format-specific file operations. It performs only one image change: saving the supplied color image as grayscale with OpenCV.


## Before You Run

In the course JupyterLite site, choose **Add files**, select `study_workspace.png`, and enter `assets/06_image_files` for **Save folder**. The site creates missing folders and stores the file as `/home/pyodide/assets/06_image_files/study_workspace.png`.

An upload-window relative folder starts at `/home/pyodide`. A relative path inside Python starts at the current working directory set in the first code cell.


## Problem Definition

### Question

How can a weekly study record be written to several useful file formats and then reopened to verify every saved result?

### Prepared Case Data

The case contains three study records, a small summary dictionary, and the supplied `assets/06_image_files/study_workspace.png` image.

### Constraints

- Use `os.path` for paths and `outputs/hands_on_study_archive` for saved results.
- Save UTF-8 text, CSV, and JSON with their standard-library tools.
- Use pickle only for trusted, program-owned binary data.
- Use OpenCV to save one grayscale PNG; do not resize, crop, or draw on it.
- Reopen all five outputs and keep repeated runs deterministic.


## Session Method Map

| Guided topic | Integrated use |
|---|---|
| Directories and paths | Build input and output paths with `os.path` |
| Text files | Write and reopen a readable weekly report |
| CSV files | Write and reopen tabular study rows |
| JSON files | Write and reopen the summary dictionary |
| Pickle files | Dump and load a trusted Python snapshot |
| Image files | Load a supplied image and save a grayscale copy with OpenCV |


## Program Flow

`locate input → create output directory → save five files → reopen five files → verify results`


## Imports, Directories, and Prepared Data

Set the JupyterLite base, locate the uploaded image asset, and create the visible archive directory. The tabular records are small enough to keep directly in the notebook.


In [1]:
import csv
import json
import os
import pickle

import cv2

jupyterlite_base_directory = "/home/pyodide"
chapter_name = "01-3_Python_Basics_III"
launch_directory = os.getcwd()

if os.path.isdir(jupyterlite_base_directory):
    base_directory = jupyterlite_base_directory
elif os.path.basename(launch_directory) == chapter_name:
    base_directory = launch_directory
else:
    base_directory = os.path.join(launch_directory, chapter_name)

if not os.path.isdir(base_directory):
    raise FileNotFoundError(f"Base directory not found: {base_directory}")

os.chdir(base_directory)

input_directory = os.path.join(base_directory, "assets", "06_image_files")
input_path = os.path.join(input_directory, "study_workspace.png")

assert os.path.isfile(input_path), (
    "Add study_workspace.png and set Save folder to assets/06_image_files."
)

image_input_path = input_path

output_directory = os.path.join(
    base_directory,
    "outputs",
    "hands_on_study_archive",
)
os.makedirs(output_directory, exist_ok=True)

study_records = [
    {"day": "Monday", "topic": "Python", "minutes": 60},
    {"day": "Tuesday", "topic": "CSV", "minutes": 45},
    {"day": "Wednesday", "topic": "JSON", "minutes": 75},
]
study_summary = {
    "week": 3,
    "sessions": len(study_records),
    "total_minutes": sum(record["minutes"] for record in study_records),
}

print("JupyterLite base:", jupyterlite_base_directory)
print("Current directory name:", os.path.basename(os.getcwd()))
print("Image input exists:", os.path.isfile(image_input_path))
print("Output folder:", os.path.relpath(output_directory, base_directory))
print("Output directory exists:", os.path.isdir(output_directory))
print("Prepared sessions:", study_summary["sessions"])


JupyterLite base: /home/pyodide
Current directory name: 01-3_Python_Basics_III
Image input exists: True
Output folder: outputs/hands_on_study_archive
Output directory exists: True
Prepared sessions: 3


The three prepared records total 180 minutes. The only external case input is the image uploaded to the numbered folder shown above.


## Saving the Archive

Each format receives a fixed file name. Reusing those names means a repeated run replaces the same outputs instead of accumulating rows or extra files.


In [2]:
def save_archive(records, summary, image_path, output_directory):
    os.makedirs(output_directory, exist_ok=True)

    output_paths = {
        "report": os.path.join(output_directory, "weekly_report.txt"),
        "table": os.path.join(output_directory, "study_records.csv"),
        "summary": os.path.join(output_directory, "study_summary.json"),
        "snapshot": os.path.join(output_directory, "study_snapshot.pkl"),
        "image": os.path.join(output_directory, "study_workspace_grayscale.png"),
    }

    report_text = (
        f"Week: {summary['week']}\n"
        f"Sessions: {summary['sessions']}\n"
        f"Total minutes: {summary['total_minutes']}\n"
    )
    with open(output_paths["report"], mode="w", encoding="utf-8") as report_file:
        report_file.write(report_text)

    with open(output_paths["table"], mode="w", encoding="utf-8", newline="") as csv_file:
        writer = csv.DictWriter(csv_file, fieldnames=["day", "topic", "minutes"])
        writer.writeheader()
        writer.writerows(records)

    with open(output_paths["summary"], mode="w", encoding="utf-8") as json_file:
        json.dump(summary, json_file, indent=2, ensure_ascii=False)

    trusted_snapshot = {"records": records, "summary": summary}
    with open(output_paths["snapshot"], mode="wb") as pickle_file:
        pickle.dump(trusted_snapshot, pickle_file)

    color_image = cv2.imread(image_path, cv2.IMREAD_COLOR)
    if color_image is None:
        raise ValueError(f"OpenCV could not load: {os.path.basename(image_path)}")
    grayscale_image = cv2.cvtColor(color_image, cv2.COLOR_BGR2GRAY)
    if not cv2.imwrite(output_paths["image"], grayscale_image):
        raise OSError(f"OpenCV could not save: {os.path.basename(output_paths['image'])}")

    return output_paths


archive_paths = save_archive(
    study_records,
    study_summary,
    image_input_path,
    output_directory,
)

print("Saved files:", sorted(os.path.basename(path) for path in archive_paths.values()))


Saved files: ['study_records.csv', 'study_snapshot.pkl', 'study_summary.json', 'study_workspace_grayscale.png', 'weekly_report.txt']


The function creates five outputs. Text, CSV, and JSON use text mode, pickle uses binary mode, and OpenCV writes the grayscale image array.


## Reopening and Verifying Results

Load every saved format with the matching reader. The pickle is safe to load here only because `save_archive()` created it in the same program-owned output directory.


In [3]:
def load_archive(output_paths):
    with open(output_paths["report"], mode="r", encoding="utf-8") as report_file:
        report = report_file.read()

    with open(output_paths["table"], mode="r", encoding="utf-8", newline="") as csv_file:
        table_rows = list(csv.DictReader(csv_file))

    with open(output_paths["summary"], mode="r", encoding="utf-8") as json_file:
        summary = json.load(json_file)

    # Safe here because save_archive() created this pickle.
    with open(output_paths["snapshot"], mode="rb") as pickle_file:
        snapshot = pickle.load(pickle_file)

    grayscale_image = cv2.imread(output_paths["image"], cv2.IMREAD_GRAYSCALE)
    if grayscale_image is None:
        raise ValueError("OpenCV could not reload the grayscale output")

    return {
        "report": report,
        "table_rows": table_rows,
        "summary": summary,
        "snapshot": snapshot,
        "image": grayscale_image,
    }


reloaded_archive = load_archive(archive_paths)

print("Report contains total:", "Total minutes: 180" in reloaded_archive["report"])
print("CSV data rows:", len(reloaded_archive["table_rows"]))
print("JSON matches:", reloaded_archive["summary"] == study_summary)
print(
    "Pickle matches:",
    reloaded_archive["snapshot"] == {
        "records": study_records,
        "summary": study_summary,
    },
)
print("Grayscale dimensions:", reloaded_archive["image"].shape)


Report contains total: True
CSV data rows: 3
JSON matches: True
Pickle matches: True
Grayscale dimensions: (600, 900)


All five files reopen successfully. The CSV contains three data rows, JSON and pickle match their source values, and the image reopens as a one-channel 600 by 900 array.


## Repeat-Run Check

Save the same archive again, then check the exact filenames and the reopened CSV row count.


In [4]:
archive_paths = save_archive(
    study_records,
    study_summary,
    image_input_path,
    output_directory,
)
repeated_archive = load_archive(archive_paths)

expected_names = {
    "weekly_report.txt",
    "study_records.csv",
    "study_summary.json",
    "study_snapshot.pkl",
    "study_workspace_grayscale.png",
}
actual_names = set(os.listdir(output_directory))

print("Exact five files:", actual_names == expected_names)
print("CSV rows after repeat save:", len(repeated_archive["table_rows"]))


Exact five files: True
CSV rows after repeat save: 3


The archive still contains five fixed filenames and three CSV data rows, so repeating the workflow does not accumulate data.


## Result Analysis

The verified archive contains exactly five files. Its report states 180 total minutes, its CSV has three records, its JSON and trusted pickle match the prepared Python data, and its OpenCV output is a 600 by 900 grayscale image. The repeat-run check confirms deterministic file replacement.


## Program Extension Points

A later project could add more weekly records, select filenames by date, or place the verified archive in a permanent user-chosen directory. Those extensions should retain the same separation between supplied inputs and generated outputs.
